# RP-010 — paired generic versus rich supervision

Read `docs/RP010_CONTROLLED_TRAINING_AUDIT.md` and `docs/RP010_RUNBOOK.md` first.
A = class + two fixed RP-009 templates; B = class + saved detailed/generalized.
Class-name counting on standard FSC147 test is primary. FSC147-S is secondary
and shares images with training. No test-driven checkpoint selection.

Source is fetched from GitHub `rich-prompt-alignment` after researcher review,
commit and push. No source ZIP. This notebook does not commit, push, start full
training by default, or disconnect the runtime. Prefer an L4 runtime; its peak
memory requirement is not yet measured.

Run all cells from top to bottom. First prepare once, then smoke A and B in
separate sessions, review both reports, then train each arm separately. Resume
replays an interrupted epoch from the last complete boundary. Keep source,
packages, GPU type and immutable asset locations identical within a pair.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, re, shutil, zipfile

ASSET_ROOT = Path('/content/drive/MyDrive/T2ICount-assets')
ASSET_ZIP = Path('/content/drive/MyDrive/T2ICount-assets.zip')  # optional existing data archive
LOCAL_DATA_ROOT = Path('/content/RP010-data/datasets/FSC147')
DATA_ROOT = None  # optional explicit existing FSC147 dataset; otherwise detect/extract dataset only
TRAIN_BANK = ASSET_ROOT / 'prompts/fsc147_train1000_seed3407_gemma4_v3.json'
S_BANK = ASSET_ROOT / 'prompts/fsc147s_gemma4_26b_v3.json'
REPO = Path('/content/T2ICount-RP010')
REPO_URL = 'https://github.com/bqa100507-spec/T2ICount-Implementation.git'
SOURCE_BRANCH = 'rich-prompt-alignment'
EXPECTED_COMMIT = ''  # optional reviewed full SHA; empty pins the branch tip for this session
SOURCE_REVISION = None

STAGE = 'prepare'  # prepare / smoke / train / evaluate / compare
ARM = 'A'          # switch A -> B; preparation is shared
SEED = 3407       # new seeds create new shared initialization
PAIR_ID = 'pair_v1'  # use a new ID for a new pair/smoke attempt; never overwrite
RUN_STAGE = False
ENABLE_FULL_TRAINING = False
RESUME = False    # train only; latest completed-epoch checkpoint in this arm's output
INSTALL_DEPENDENCIES = True

assert STAGE in ('prepare','smoke','train','evaluate','compare') and ARM in ('A','B')
assert re.fullmatch(r'[A-Za-z0-9_-]+', PAIR_ID)
assert not RESUME or STAGE == 'train'

def stream(command, cwd=None):
    print('+', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(list(map(str,command)), cwd=cwd, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait():
        raise RuntimeError(f'Command failed with exit {process.returncode}; read the first error above')

def git(*args):
    return subprocess.check_output(['git','-C',str(REPO),*args],text=True).strip()


## Runtime and Drive
An available GPU alone is not a smoke pass. Preparation checks host RAM;
smoke checks a model-derived memory budget estimate before allocating GPU state.
The estimate excludes activations and optimizer workspace. Do not alter precision,
crop size, freezing or losses to fit this comparison onto a smaller GPU.


In [ ]:
if RUN_STAGE:
    import torch
    print('Python:', sys.version)
    print('Torch/CUDA:', torch.__version__, torch.version.cuda)
    assert torch.cuda.is_available(), 'Choose a Colab GPU runtime; prefer L4'
    assert torch.cuda.device_count() == 1, 'RP-010 uses exactly one GPU'
    print('GPU:', torch.cuda.get_device_name(0))
    print('Free/total GiB:', [round(x/2**30,2) for x in torch.cuda.mem_get_info()])
    if not Path('/content/drive/MyDrive').is_dir():
        from google.colab import drive
        drive.mount('/content/drive')
    assert ASSET_ROOT.is_dir(), f'Missing external assets: {ASSET_ROOT}'
else:
    print('Safe default: RUN_STAGE=False; no setup, GPU job or training launched.')


## Reviewed GitHub source
A notebook already open in VS Code/Colab does not update when GitHub changes.
Reopen the reviewed notebook if it contains old cells. Existing modified source
is rejected; no local work is reset. Pin the full SHA for repeat runs/resume.


In [ ]:
if RUN_STAGE:
    if not REPO.exists():
        stream(['git','clone','--single-branch','--branch',SOURCE_BRANCH,REPO_URL,REPO])
    assert git('remote','get-url','origin').removesuffix('.git') == REPO_URL.removesuffix('.git')
    assert git('branch','--show-current') == SOURCE_BRANCH
    assert not git('status','--porcelain','--untracked-files=no'), 'Modified source; preserve it and use a fresh checkout'
    stream(['git','-C',REPO,'fetch','origin',SOURCE_BRANCH])
    remote = git('rev-parse',f'origin/{SOURCE_BRANCH}')
    if EXPECTED_COMMIT:
        assert re.fullmatch(r'[0-9a-f]{40}',EXPECTED_COMMIT)
        assert remote == EXPECTED_COMMIT, 'Remote tip differs from the reviewed SHA'
    stream(['git','-C',REPO,'merge','--ff-only',f'origin/{SOURCE_BRANCH}'])
    SOURCE_REVISION = git('rev-parse','HEAD')
    assert SOURCE_REVISION == remote
    required = ['tools/train_rp010.py','utils/rp010.py','configs/rp010a.json',
                'configs/rp010b.json','docs/RP010_CONTROLLED_TRAINING_AUDIT.md','docs/RP010_RUNBOOK.md']
    assert all((REPO/p).is_file() for p in required), 'RP-010 files are not on GitHub yet: review, commit and push them first'
    print('Pinned source:',SOURCE_REVISION)


## Dependencies and immutable assets
Uses the repository's pinned Colab requirements and taming-transformers import
setup. No replacement Torch/CUDA stack. If installation requests a runtime
restart, restart and rerun from the beginning before preparing the initialization.
Dataset-only extraction below is optional; model weights remain on Drive.


In [ ]:
if RUN_STAGE:
    if INSTALL_DEPENDENCIES:
        stream([sys.executable,'-m','pip','install','-r',REPO/'requirements-colab.txt'],cwd=REPO)
    taming = next(iter(REPO.glob('src/taming-transformers')),None)
    assert taming is not None, 'Pinned editable taming-transformers checkout missing'
    os.environ['PYTHONPATH'] = os.pathsep.join([str(REPO),str(taming),os.environ.get('PYTHONPATH','')])
    for location in (REPO,taming):
        if str(location) not in sys.path:
            sys.path.insert(0,str(location))
    stream([sys.executable,'-c','import torch, torchvision, transformers, taming; from tools.train_rp010 import ControlledTrainer; print("Imports OK")'],cwd=REPO)

    if DATA_ROOT is None:
        for candidate in (LOCAL_DATA_ROOT,ASSET_ROOT/'datasets/FSC147'):
            if (candidate/'FSC_147/Train_Test_Val_FSC_147.json').is_file():
                DATA_ROOT = candidate
                break
    if DATA_ROOT is None:
        assert ASSET_ZIP.is_file(), 'Set DATA_ROOT to the existing extracted FSC147 assets, or set ASSET_ZIP'
        prefix = ('datasets','FSC147')
        extracted = 0
        with zipfile.ZipFile(ASSET_ZIP) as archive:
            for member in archive.infolist():
                parts = Path(member.filename).parts
                for start in range(len(parts)-1):
                    if tuple(parts[start:start+2]) == prefix:
                        relative = Path(*parts[start+2:])
                        assert not relative.is_absolute() and '..' not in relative.parts
                        target = LOCAL_DATA_ROOT / relative
                        if member.is_dir():
                            target.mkdir(parents=True,exist_ok=True)
                        else:
                            assert not target.exists(), f'Partial cache exists: {target}; use a fresh LOCAL_DATA_ROOT'
                            target.parent.mkdir(parents=True,exist_ok=True)
                            with archive.open(member) as source, target.open('wb') as destination:
                                shutil.copyfileobj(source,destination)
                            from datetime import datetime
                            stamp = datetime(*member.date_time).timestamp()
                            os.utime(target,(stamp,stamp))  # stable identity across runtime extraction
                            extracted += 1
                            if extracted % 250 == 0:
                                print('Extracted dataset files:',extracted,flush=True)
                        break
        DATA_ROOT = LOCAL_DATA_ROOT
    for path in (DATA_ROOT/'FSC_147/Train_Test_Val_FSC_147.json', TRAIN_BANK,
                 ASSET_ROOT/'pretrained/sd-v1-5/v1-5-pruned-emaonly.ckpt',
                 ASSET_ROOT/'pretrained/clip-vit-large-patch14/config.json'):
        assert path.is_file(), f'Missing required asset: {path}'
    print('Dataset:',DATA_ROOT,'\nTraining bank:',TRAIN_BANK)


## Execute the selected stage
`prepare` creates one shared initialization and token audit. `smoke` runs one
compatible three-forward batch, real loss/backward/update, save/load, one-image
validation and a second resumed update. Run it once per arm; read both results.

`train` requires RUN_STAGE=True **and** ENABLE_FULL_TRAINING=True plus both
matching passed smoke reports. Checkpoints/logs are saved to Drive every completed
epoch. If interrupted before the first checkpoint, preserve that folder and use
a new PAIR_ID; otherwise set RESUME=True. A partial epoch is replayed.

`evaluate` accepts only completed runs and their validation-selected best weights;
it produces separate standard-test and FSC147-S class tables. `compare` verifies
paired control traces and joins per-image errors. It does not choose a winner.


In [ ]:
if RUN_STAGE:
    assert SOURCE_REVISION and git('rev-parse','HEAD') == SOURCE_REVISION
    assert not git('status','--porcelain','--untracked-files=no')
    command = [sys.executable,'-u',REPO/'tools/train_rp010.py',STAGE,
               '--config',REPO/f'configs/rp010{ARM.lower()}.json',
               '--asset-root',ASSET_ROOT,'--seed',SEED,'--pair-id',PAIR_ID,
               '--data-dir',DATA_ROOT,'--prompt-bank',TRAIN_BANK]
    if STAGE == 'train':
        assert ENABLE_FULL_TRAINING, 'Review audit and both smokes, then explicitly enable full training'
        command += ['--allow-full-training']
        if RESUME:
            command += ['--resume']
    if STAGE == 'evaluate':
        assert S_BANK.is_file(), f'Missing existing evaluation bank: {S_BANK}'
        command += ['--s-metadata',REPO/'FSC-147-S.json','--s-bank',S_BANK]
    if STAGE == 'compare':
        command += ['--comparison-output',ASSET_ROOT/f'outputs/rp010/seed{SEED}/{PAIR_ID}/paired_analysis']
    stream(command,cwd=REPO)


## Inspect artifacts before the next stage
Do not infer success from a notebook cell merely finishing. Missing/pending smoke,
failed validation, unexpected step count or changed source/assets blocks training.
Full-run completion is 10 epochs and 10,000 optimizer steps per arm, selected only
by validation MAE+RMSE at epochs7/8/9. Conditional bootstrap intervals describe
image uncertainty for this pair; replicate training seeds before stable claims.


In [ ]:
if RUN_STAGE:
    run = ASSET_ROOT/f'outputs/rp010/seed{SEED}/{PAIR_ID}/RP-010{ARM}'
    if STAGE == 'prepare':
        report = ASSET_ROOT/f'checkpoints/rp010/init_seed{SEED}/initial_manifest.json'
        info = json.loads(report.read_text())
        print('Init SHA:',info['initialization_sha256'])
        print('GPU memory budget estimate (GiB):',info['memory']['weights_gradients_adam_estimate_bytes']/2**30)
        print('Token audit:',report.parent/'prompt_tokens.csv')
    elif STAGE == 'smoke':
        report = run.with_name(run.name+'_smoke')/'smoke_result.json'
        info = json.loads(report.read_text())
        print(json.dumps({k:v for k,v in info.items() if k not in ('guard','data')},indent=2))
        assert info['status'] == 'passed', 'GPU smoke is pending/failed; do not launch full training'
    elif STAGE == 'train':
        info = json.loads((run/'completed.json').read_text())
        assert info['epochs'] == 10 and info['optimizer_steps'] == 10000
        selected = run/info['selection']['checkpoint']
        assert selected.is_file() and (run/'latest.tar').is_file()
        print('Completed; selected by validation:',info['selection'])
    elif STAGE == 'evaluate':
        print((run/'evaluation_class/summary.json').read_text())
    else:
        print((run.parent/'paired_analysis/summary.json').read_text())
